# 🤖 SAHAYAK JARVIS — Self-Evolution Factory (Colab T4)
One-click: **Teacher (Llama-405B) → Synthetic Data → LoRA → Eval → Quantize → OTA**

1. Upload `seed_instructions.jsonl` + `user_logs.jsonl` (exported from app: Settings → Export)
2. Set `HF_TOKEN` in Secrets (for Llama-405B via HuggingFace Inference)
3. Run All → download `jarvis-lora-merged-q4.gguf` → drop in `app_android/app/src/main/assets/models/`


In [ ]:
# Install
!pip -q install transformers==4.44.2 datasets==3.0.1 trl==0.9.6 peft==0.12.0 accelerate==0.33.0 bitsandbytes==0.43.3 huggingface_hub==0.24.6 llama-cpp-python optimum
import os, json, random
from pathlib import Path
from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])  # set via Colab Secrets

In [ ]:
# 1) Distill from Teacher — Llama-3.1-405B-Instruct
teacher_model = "meta-llama/Meta-Llama-3.1-405B-Instruct"  # via HF Inference / Together API
student_model = "HuggingFaceTB/SmolLM2-360M-Instruct"

# If you don't have 405B quota, use 70B — still great:
# teacher_model = "meta-llama/Meta-Llama-3.1-70B-Instruct"

from training.scripts.distill import generate_dataset
ds = generate_dataset(
    seed_path="../datasets/seed_instructions.jsonl",
    user_logs_path="../datasets/user_logs.jsonl",  # optional
    teacher=teacher_model,
    n_samples=5000,
    out_path="../datasets/jarvis_synthetic.jsonl"
)
print(f"Generated {len(ds)} samples")


In [ ]:
# 2) Fine-tune small model with QLoRA
!python ../scripts/finetune.py \
  --base_model HuggingFaceTB/SmolLM2-360M-Instruct \
  --dataset ../datasets/jarvis_synthetic.jsonl \
  --output_dir ./jarvis-lora \
  --lora_r 16 --lora_alpha 32 \
  --epochs 2 --batch_size 4 --grad_accum 4 \
  --lr 2e-4 --use_4bit

In [ ]:
# 3) Eval — tool accuracy + personality
!python ../scripts/eval.py --model ./jarvis-lora --test ../datasets/holdout.jsonl
# Expect: tool_call_accuracy > 0.85, personality_score > 4.2/5

In [ ]:
# 4) Merge + quantize to GGUF Q4_K_M for phone
!python ../scripts/merge_and_quantize.py --lora ./jarvis-lora --base HuggingFaceTB/SmolLM2-360M-Instruct --out ./jarvis-merged
!python -m llama_cpp.convert --outtype q4_k_m ./jarvis-merged jarvis-q4_k_m.gguf
!ls -lh *.gguf
from google.colab import files
files.download("jarvis-q4_k_m.gguf")